# 04 · ChemKAN actual implementation walkthrough

This notebook is for **understanding and validating the real implementation**, not rebuilding ChemKAN again.

Use the notebooks differently:

- `03_chemkan_from_scratch_paper_aligned.ipynb` → learn the architecture and equations.
- **this notebook** → import the real `src/chemkan/` code and check that it behaves as expected.

We deliberately use only **2 biodiesel trajectories** and initially only **3 saved time points**.

Study rule: before running each code cell, predict the important tensor shapes.

Main flow:

$$
\text{data}\rightarrow[Y,T]\rightarrow\text{normalizer}\rightarrow KAN_{kin}
\rightarrow\dot Y\rightarrow\text{ODE RHS}\rightarrow\text{ODE solver}
\rightarrow\hat Y(t)\rightarrow L\rightarrow\nabla_\theta L.
$$

## Step 0 · Use the same paths as your current notebook

This notebook assumes it sits in the same directory as your current walkthrough, so we use the paths directly:

```text
../data/generated/biodiesel.npz
../src/chemkan/
```

In [1]:
from pathlib import Path
import sys
import inspect
import numpy as np
import torch

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

DATA_PATH = Path('../data/generated/biodiesel.npz')
SRC_PATH = Path('../src')

assert DATA_PATH.exists(), f'Missing data: {DATA_PATH.resolve()}'
assert (SRC_PATH / 'chemkan').exists(), f'Missing source: {(SRC_PATH / "chemkan").resolve()}'

sys.path.insert(0, str(SRC_PATH.resolve()))

print('data:', DATA_PATH.resolve())
print('src :', SRC_PATH.resolve())

data: /Users/berke/Desktop/University/TU_Wien-MSC/2026S/Interdisciplinary_Project/development/chemkan-tuwien/chemkan/data/generated/biodiesel.npz
src : /Users/berke/Desktop/University/TU_Wien-MSC/2026S/Interdisciplinary_Project/development/chemkan-tuwien/chemkan/src


In [2]:
from chemkan.model import KineticCore, ThermodynamicSuperstructure, ChemKAN
from chemkan.normalization import MinMaxNormalizer
from chemkan.temperature import ConstantTemperature, ObservedTemperature
from chemkan.dynamics import KineticDynamics, ChemKANDynamics
from chemkan.solver import SolverConfig, integrate
from chemkan.losses import trajectory_mse, element_conservation_loss, chemkan_loss
from chemkan.training import train_kinetic_stage, train_full_chemkan

print('KineticCore     :', inspect.getfile(KineticCore))
print('KineticDynamics :', inspect.getfile(KineticDynamics))
print('integrate       :', inspect.getfile(integrate))
print('trajectory_mse  :', inspect.getfile(trajectory_mse))

KineticCore     : /Users/berke/Desktop/University/TU_Wien-MSC/2026S/Interdisciplinary_Project/development/chemkan-tuwien/chemkan/src/chemkan/model.py
KineticDynamics : /Users/berke/Desktop/University/TU_Wien-MSC/2026S/Interdisciplinary_Project/development/chemkan-tuwien/chemkan/src/chemkan/dynamics.py
integrate       : /Users/berke/Desktop/University/TU_Wien-MSC/2026S/Interdisciplinary_Project/development/chemkan-tuwien/chemkan/src/chemkan/solver.py
trajectory_mse  : /Users/berke/Desktop/University/TU_Wien-MSC/2026S/Interdisciplinary_Project/development/chemkan-tuwien/chemkan/src/chemkan/losses.py


### MY NOTES

- Which file owns the architecture?
- Which file owns the physical-state → model-input bridge?
- Which file owns integration?

# Part I · Data → physical state → model input

## Step 1 · Inspect the `.npz`

An `.npz` is just an archive of named NumPy arrays. `data.files` shows the stored names.

In [3]:
data = np.load(DATA_PATH, allow_pickle=True)

for key in data.files:
    arr = data[key]
    print(f'{key:24s} shape={getattr(arr, "shape", None)}')

t                        shape=(30,)
species                  shape=(6,)
mechanism                shape=()
state_layout             shape=()
train_states             shape=(20, 30, 6)
test_states              shape=(10, 30, 6)
train_T                  shape=(20,)
test_T                   shape=(10,)
u_min                    shape=(6,)
u_max                    shape=(6,)
true_Ea_kcal             shape=(3,)
true_lnA                 shape=(3,)
train_states_noise00     shape=(20, 30, 6)
test_states_noise00      shape=(10, 30, 6)
train_states_noise01     shape=(20, 30, 6)
test_states_noise01      shape=(10, 30, 6)
train_states_noise02     shape=(20, 30, 6)
test_states_noise02      shape=(10, 30, 6)
train_states_noise05     shape=(20, 30, 6)
test_states_noise05      shape=(10, 30, 6)
train_states_noise07     shape=(20, 30, 6)
test_states_noise07      shape=(10, 30, 6)
train_states_noise10     shape=(20, 30, 6)
test_states_noise10      shape=(10, 30, 6)
train_states_noise15     shape=(20, 30,

## Step 2 · Load only two biodiesel trajectories

The generated array starts as

$$
(B,T,m)=(20,30,6).
$$

For ODE work we use time-first notation

$$
(T,B,m).
$$

In [69]:
# corresponds to _data.py
species = [str(s) for s in data['species']]
m = len(species)

states_BTm = torch.as_tensor(data['train_states'], dtype=torch.float32)  # (20,30,6)
t = torch.as_tensor(data['t'], dtype=torch.float32)                     # (30,)
T_all = torch.as_tensor(data['train_T'], dtype=torch.float32)           # (20,)

B_SMALL = 2
states = states_BTm[:B_SMALL].permute(1,0,2).contiguous()               # (30,2,6)
T_const = T_all[:B_SMALL]                                                # (2,)
Y0 = states[0]                                                           # (2,6)

print('species     :', species)
print('states_BTm  :', states_BTm.shape)
print('states      :', states.shape)
print('t           :', t.shape)
print('T_const     :', T_const.shape)
print('Y0          :', Y0.shape)
print('\nstates[0,0,:] =', states[0,0,:])
print('states[0,0,0] =', states[0,0,0], '<- TG only')

species     : ['TG', 'ROH', 'DG', 'MG', 'GL', 'RCO2R']
states_BTm  : torch.Size([20, 30, 6])
states      : torch.Size([30, 2, 6])
t           : torch.Size([30])
T_const     : torch.Size([2])
Y0          : torch.Size([2, 6])

states[0,0,:] = tensor([1.4554, 0.5425, 0.0000, 0.0000, 0.0000, 0.0000])
states[0,0,0] = tensor(1.4554) <- TG only


### CHECKPOINT 1

Explain:

$$
(20,30,6)\rightarrow(30,2,6)\rightarrow Y_0:(2,6).
$$

Also explain why `states[0,0,:]` is all six species, not TG alone.

### MY NOTES

Because TG is states[0,0,0]. However, states[0,0,:] is t=0
and states became [30,2,6] so the data is grouped per time step, states[0,:,:] this gives all t=0.

## Step 3 · Append temperature

The biodiesel ODE integrates only the six species, but the kinetic core receives

$$
u=[TG,ROH,DG,MG,GL,RCO2R,T].
$$

In [16]:
print(T_const)
print(T_const.unsqueeze(-1))

tensor([334.4306, 329.4374])
tensor([[334.4306],
        [329.4374]])


In [5]:
T0 = T_const.unsqueeze(-1)                    # (2,1)
u0_physical = torch.cat([Y0, T0], dim=-1)    # (2,7)

print('Y0          :', Y0.shape)
print('T0          :', T0.shape)
print('u0_physical :', u0_physical.shape)

for name, value in zip(species + ['T'], u0_physical[0]):
    print(f'{name:8s}: {value.item():.6f}')

Y0          : torch.Size([2, 6])
T0          : torch.Size([2, 1])
u0_physical : torch.Size([2, 7])
TG      : 1.455443
ROH     : 0.542480
DG      : 0.000000
MG      : 0.000000
GL      : 0.000000
RCO2R   : 0.000000
T       : 334.430603


### CHECKPOINT 2

Why is the biodiesel ODE state 6-dimensional while `KineticCore` receives 7 values?

### MY NOTES

Because we concat T and the states so it became m species and T so for biodiesel it is m+1 = 7

## Step 4 · Build the same full-state normalizer as `train_biodiesel.py`

The `.npz` stores train-only species min/max values. The production training script appends train-temperature min/max to create a normalizer for all 7 kinetic inputs.

Keep the representations separate:

$$
u_{physical}\rightarrow u_{model}\rightarrow\tanh(u_{model})\rightarrow RBF.
$$

The ODE state itself remains physical.

In [20]:
u_min_species = torch.as_tensor(data['u_min'], dtype=torch.float32)
u_max_species = torch.as_tensor(data['u_max'], dtype=torch.float32)

u_min_full = torch.cat([u_min_species, T_all.min().reshape(1)])
u_max_full = torch.cat([u_max_species, T_all.max().reshape(1)])

full_norm = MinMaxNormalizer(u_min_full, u_max_full)
loss_norm = full_norm.subset(slice(0, m))

u0_model = full_norm.normalize(u0_physical)

print('u_min_full:', u_min_full)
print('u_max_full:', u_max_full)
print('\nphysical T    :', u0_physical[:,-1])
print('scaled T      :', u0_model[:,-1])
print('tanh(raw T)   :', torch.tanh(u0_physical[:,-1]))
print('tanh(scaled T):', torch.tanh(u0_model[:,-1]))

u_min_full: tensor([  0.1804,   0.0107,   0.0000,   0.0000,   0.0000,   0.0000, 324.0404])
u_max_full: tensor([  1.9026,   1.9958,   0.3217,   0.3551,   0.3194,   1.8600, 340.8055])

physical T    : tensor([334.4306, 329.4374])
scaled T      : tensor([0.6198, 0.3219])
tanh(raw T)   : tensor([1., 1.])
tanh(scaled T): tensor([0.5510, 0.3112])


### CHECKPOINT 3

Explain the difference between:

- physical ODE state,
- pre-KAN min-max model input,
- KAN-internal `tanh` input.

### MY NOTES
When we implement min-max normalization before data entering the model, the output of tanh(T) doesn't saturate as we can see from the outsputs of tanh(raw T) and tanh(scaled T).

# Part II · Verify the actual kinetic core

## Step 5 · Instantiate the paper-sized biodiesel core

$$
7\rightarrow4\rightarrow6,\qquad N=3,\qquad n^\mu=2.
$$

The current base-off count-matching kinetic core should have 156 trainable parameters.

In [21]:
core = KineticCore(
    species_dim=6,
    hidden_dim=4,
    num_basis=3,
    n_mu=2,
    use_base_act=False,
)

n_params = sum(p.numel() for p in core.parameters() if p.requires_grad)

print(core)
print('\nparams:', n_params)
print('first w_rbf :', core.add.edges.w_rbf.shape)
print('second w_rbf:', core.lean.edges.w_rbf.shape)

assert n_params == 156
assert core.add.edges.w_rbf.shape == (4,7,3)
assert core.lean.edges.w_rbf.shape == (6,4,3)

KineticCore(
  (add): AddKANLayer(
    (edges): RBFEdgeFunctions()
  )
  (lean): LeanKANLayer(
    (edges): RBFEdgeFunctions()
  )
)

params: 156
first w_rbf : torch.Size([4, 7, 3])
second w_rbf: torch.Size([6, 4, 3])


`w_rbf.shape == (4,7,3)` means:

```text
4 destination hidden nodes
× 7 source nodes
× 3 RBF weights per edge
```

There is no batch dimension in the learned weights.

## Step 6 · Quick manual check: TG → hidden node 3

Paper edge:

$$
\phi_{0,3,1}.
$$

Zero-based PyTorch indices: `o=2, i=0`.

In [116]:
edge0[0,2,:]

tensor([ 0.0333, -0.1720,  0.0972, -0.0493, -0.0469,  0.3908,  0.0001],
       grad_fn=<SelectBackward0>)

In [127]:
u0_model.shape, edge0.shape

(torch.Size([2, 7]), torch.Size([2, 4, 7]))

In [72]:
edge0 = core.add.edges(u0_model) # the result of psi*w_rbf all of the edges from input nodes to hidden nodes for two batch where t=0.

x_internal = torch.tanh(u0_model) # u0_model is the min max normalized version of u0_physical
centers = core.add.edges.centers
h = core.add.edges.h
w0 = core.add.edges.w_rbf

psi0 = torch.exp(-(x_internal.unsqueeze(-1)-centers)**2 / (2*h**2))

b,o,i = 0,2,0
manual_edge = (psi0[b,i,:] * w0[o,i,:]).sum()
actual_edge = edge0[b,o,i]

print('psi TG     :', psi0[b,i,:])
print('w TG -> H3 :', w0[o,i,:])
print('manual     :', manual_edge.item())
print('actual     :', actual_edge.item())

assert torch.allclose(manual_edge, actual_edge, atol=1e-6)

psi TG     : tensor([0.2652, 0.8203, 0.9336])
w TG -> H3 : tensor([-0.0350, -0.0007,  0.0463], grad_fn=<SelectBackward0>)
manual     : 0.033331215381622314
actual     : 0.033331211656332016


This checks

$$
E^{(0)}_{b,o,i}=\sum_{k=1}^{3}\psi_{b,i,k}w^{(0)}_{o,i,k}.
$$

You already understand this part, so after the assertion passes, move on.

## Step 7 · Verify AddKAN

After the RBF contraction:

$$
edge0:(B,4,7).
$$

For hidden node 3, `edge0[0,2,:]` contains the seven incoming edge values.

In [117]:
hidden = core.add(u0_model)
incoming_H3 = edge0[0,2,:]
manual_H3 = incoming_H3.sum()

print('incoming H3:', incoming_H3)
print('manual H3  :', manual_H3.item())
print('actual H3  :', hidden[0,2].item())

assert torch.allclose(manual_H3, hidden[0,2], atol=1e-6)
assert torch.allclose(hidden, edge0.sum(dim=-1), atol=1e-6)

incoming H3: tensor([ 0.0333, -0.1720,  0.0972, -0.0493, -0.0469,  0.3908,  0.0001],
       grad_fn=<SelectBackward0>)
manual H3  : 0.2533063292503357
actual H3  : 0.2533063292503357


Remember:

$$
\sum_k\Rightarrow\text{one edge},\qquad
\sum_i\Rightarrow\text{one AddKAN node}.
$$

## Step 8 · Verify LeanKAN

The second layer creates

$$
edge1:(B,6,4).
$$

With `n_mu=2`, for one output:

$$
z_o=E_{o,0}E_{o,1}+E_{o,2}+E_{o,3}.
$$

In [188]:
test = torch.tanh(hidden)
test_h = 1

print("shape of test before unsqueeze and subtracting from centers: ", test.shape)
print("shape of test after unsqueeze: ", test.unsqueeze(-1).shape)
test = test.unsqueeze(-1) - torch.tensor([-1.0, 0., 1.])
print("shape of test after unsqueeze and subtracting centers: ", test.shape) # 

print("shape of w_rbf at LeanKAN: ", core.lean.edges.w_rbf.shape)# (6,4,3) 6 output nodes, 4 incoming edge from hidden nodes (4 node) and 3 bumps
test_gaussian_rbf_output = torch.exp(- test ** 2 / 2 * test_h**2)
leankan_output = torch.einsum("bik,oik->boi", test_gaussian_rbf_output, core.lean.edges.w_rbf)
print("shape of leankan output: ", leankan_output.shape)
leankan_output # shape is 6 outgoing edges from each node and we have 2 batches.
# sum each of them
leankan_output[:, :, :2].prod(dim=-1) + leankan_output[:, :, 2:].sum(dim=-1) # multiply until n_mu  and sum from n_mu where n_mu is 2

shape of test before unsqueeze and subtracting from centers:  torch.Size([2, 4])
shape of test after unsqueeze:  torch.Size([2, 4, 1])
shape of test after unsqueeze and subtracting centers:  torch.Size([2, 4, 3])
shape of w_rbf at LeanKAN:  torch.Size([6, 4, 3])
shape of leankan output:  torch.Size([2, 6, 4])


tensor([[-0.0243, -0.2089,  0.0722, -0.0131, -0.1903, -0.1204],
        [-0.0232, -0.2071,  0.0750, -0.0154, -0.1741, -0.1185]],
       grad_fn=<AddBackward0>)

In [189]:
edge1 = core.lean.edges(hidden) #hidden is the addkanlayer output. 
lean_out = core.lean(hidden)

b,o = 0,2
manual_lean = edge1[b,o,:2].prod() + edge1[b,o,2:].sum()

print('edges into output 0:', edge1[b,o,:])
print('product first 2    :', edge1[b,o,:2].prod().item())
print('sum last 2         :', edge1[b,o,2:].sum().item())
print('manual             :', manual_lean.item())
print('actual             :', lean_out[b,o].item())

assert torch.allclose(manual_lean, lean_out[b,o], atol=1e-6)

edges into output 0: tensor([ 0.1504,  0.2186, -0.0437,  0.0831], grad_fn=<SelectBackward0>)
product first 2    : 0.0328671857714653
sum last 2         : 0.03934421390295029
manual             : 0.07221139967441559
actual             : 0.07221139967441559


## Step 9 · Verify the complete kinetic core

$$
KAN_{kin}=\Psi^{lean}_1\circ\Psi^{add}_0.
$$

Its six outputs are current species rates, not future concentrations.

In [187]:
manual_core = core.lean(core.add(u0_model))
actual_core = core(u0_model)

print('input :', u0_model.shape)
print('hidden:', hidden.shape)
print('dY/dt :', actual_core.shape)
print(actual_core[0])

assert actual_core.shape == (B_SMALL,6)
assert torch.allclose(manual_core, actual_core, atol=1e-6)

input : torch.Size([2, 7])
hidden: torch.Size([2, 4])
dY/dt : torch.Size([2, 6])
tensor([-0.0243, -0.2089,  0.0722, -0.0131, -0.1903, -0.1204],
       grad_fn=<SelectBackward0>)


### CHECKPOINT 4

Be able to say:

$$
KAN_{kin}([Y,T])=\dot Y.
$$

### MY NOTES

# Part III · ODE right-hand side

## Step 10 · Read the actual production forward function

In [191]:
print(inspect.getsource(KineticDynamics.forward))

    def forward(self, t: torch.Tensor, Y: torch.Tensor) -> torch.Tensor:  # (B, m)->(B, m)
        T = self.temperature(t)                              # (B, 1) physical Kelvin
        u_physical = torch.cat([Y, T], dim=-1)               # (B, m+1) physical
        u_model = (u_physical if self.input_normalizer is None
                   else self.input_normalizer.normalize(u_physical))
        return self.kinetic(u_model)                         # physical dY/dt (B, m)



Before running the next cell, predict:

```text
input Y      : (2,6)
temperature  : (2,1)
physical u   : (2,7)
model u      : (2,7)
returned dYdt: (2,6)
```

In [192]:
temperature_provider = ConstantTemperature(T_const)

dynamics = KineticDynamics(
    core,
    temperature_provider,
    input_normalizer=full_norm,
)

dYdt_rhs = dynamics(t[0], Y0)

print('Y0      :', Y0.shape)
print('dYdt_rhs:', dYdt_rhs.shape)

assert torch.allclose(dYdt_rhs, core(u0_model), atol=1e-6)

Y0      : torch.Size([2, 6])
dYdt_rhs: torch.Size([2, 6])


### CHECKPOINT 5

Explain `KineticDynamics.forward` line by line:

1. where does physical temperature come from?
2. where is `[Y,T]` constructed?
3. where is min-max scaling applied?
4. what does the returned tensor physically mean?

### MY NOTES

# Part IV · ODE integration

## Step 11 · Inspect the real solver wrapper

In [193]:
print(inspect.getsource(integrate))

def integrate(func, y0: torch.Tensor, t: torch.Tensor,
              config: SolverConfig) -> torch.Tensor:
    r"""Integrate ``dy/dt = func(t, y)`` from ``y0`` over grid ``t``.

        y0 : (B, dim)   t : (T,)   ->   (T, B, dim)

    ``config`` is required -- there is no fallback solver configuration inside the
    reusable library. Uses ``torchdiffeq.odeint`` with direct autograd (gradients flow
    through the solver); ``odeint_adjoint`` is deliberately not used.
    """
    return odeint(func, y0, t, method=config.method, rtol=config.rtol, atol=config.atol)



In [275]:
solver = SolverConfig(
    method='tsit5',
    rtol=1e-6,
    atol=1e-8,
    sensitivity='direct_autograd',
)
print(solver)

SolverConfig(method='tsit5', rtol=1e-06, atol=1e-08, sensitivity='direct_autograd')


## Step 12 · Integrate only the first 3 saved times

Expected:

$$
Y_0:(B,6)\rightarrow\hat Y:(3,B,6).
$$

In [276]:
t_small = t[:3]
pred_small = integrate(dynamics, Y0, t_small, solver)

print('Y0        :', Y0.shape)
print('t_small   :', t_small.shape)
print('pred_small:', pred_small.shape)

assert pred_small.shape == (3,B_SMALL,6)
assert torch.allclose(pred_small[0], Y0, atol=1e-6)

Y0        : torch.Size([2, 6])
t_small   : torch.Size([3])
pred_small: torch.Size([3, 2, 6])


### Connection to your earlier 30-time-point question

When studying one KAN evaluation you selected only \(t_0\), so the time axis disappeared.

Now the ODE solver builds a trajectory:

$$
Y_0:(B,6)\rightarrow\hat Y:(T,B,6).
$$

The KAN itself still receives only the **current state** each time the solver asks for a derivative.

## Step 13 · Count adaptive solver calls

In [277]:
class CountingDynamics(torch.nn.Module):
    def __init__(self, inner):
        super().__init__()
        self.inner = inner
        self.calls = 0
        self.times = []

    def forward(self, tt, yy):
        self.calls += 1
        if len(self.times) < 12:
            self.times.append(float(tt.detach().cpu()))
        return self.inner(tt, yy)

counted = CountingDynamics(dynamics)
_ = integrate(counted, Y0, t_small, solver)

print('requested output points:', len(t_small))
print('RHS calls              :', counted.calls)
print('first internal times   :', counted.times)

requested output points: 3
RHS calls              : 44
first internal times   : [0.0, 0.002124601975083351, 0.0033196331933140755, 0.006742360070347786, 0.018556954339146614, 0.020206989720463753, 0.02061883732676506, 0.02061883732676506, 0.028969481587409973, 0.037579458206892014, 0.06729944050312042, 0.07145015895366669]


### CHECKPOINT 6

Explain why 30 stored observation points do **not** imply 30 ChemKAN forward calls.

### MY NOTES

# Part V · Loss

## Step 14 · Inspect the actual trajectory MSE

In [278]:
print(inspect.getsource(trajectory_mse))

def trajectory_mse(pred_norm: torch.Tensor, target_norm: torch.Tensor) -> torch.Tensor:
    r"""Eq. 18 MSE on normalized states.  pred/target: (T, B, n*) -> scalar.

    ``n*`` (= m for Stage 1, m+1 for Stage 2) is implicit in the last axis.
    """
    if pred_norm.shape != target_norm.shape:
        raise ValueError(f"shape mismatch: {pred_norm.shape} vs {target_norm.shape}")
    per_state = ((pred_norm - target_norm) ** 2).mean(dim=-1)   # (1/n*) sum_k -> (T, B)
    per_traj = per_state.sum(dim=0)                             # sum over timesteps -> (B,)
    return per_traj.mean()                                      # mean over trajectories



For Stage 1 biodiesel, the production loss uses normalized trajectories and reduces:

```text
(T,B,m)
  ↓ mean over m species
(T,B)
  ↓ sum over time
(B,)
  ↓ mean over trajectories
scalar
```

In [279]:
target_small = states[:3]  # physical (3,2,6)

pred_norm = loss_norm.normalize(pred_small)
target_norm = loss_norm.normalize(target_small)

sq = (pred_norm-target_norm)**2
manual_loss = sq.mean(dim=-1).sum(dim=0).mean()
actual_loss = trajectory_mse(pred_norm, target_norm)

print('pred_norm :', pred_norm.shape)
print('target    :', target_norm.shape)
print('sq        :', sq.shape)
print('manual    :', manual_loss.item())
print('production:', actual_loss.item())

assert torch.allclose(manual_loss, actual_loss)

pred_norm : torch.Size([3, 2, 6])
target    : torch.Size([3, 2, 6])
sq        : torch.Size([3, 2, 6])
manual    : 0.03267478942871094
production: 0.03267478942871094


### CHECKPOINT 7

Why is loss evaluated on the **integrated trajectory** rather than directly on `dY/dt`?

### MY NOTES

# Part VI · Backpropagation and training

## Step 15 · One backward pass through the ODE solve

In [280]:
core.zero_grad(set_to_none=True)

pred = integrate(dynamics, Y0, t_small, solver)
loss = trajectory_mse(loss_norm.normalize(pred), target_norm)
loss.backward()

grad = core.add.edges.w_rbf.grad

print('loss          :', loss.item())
print('parameter     :', core.add.edges.w_rbf.shape)
print('gradient      :', None if grad is None else grad.shape)
print('gradient norm :', None if grad is None else grad.norm().item())

assert grad is not None
assert grad.shape == core.add.edges.w_rbf.shape
assert torch.isfinite(grad).all()

loss          : 0.03267478942871094
parameter     : torch.Size([4, 7, 3])
gradient      : torch.Size([4, 7, 3])
gradient norm : 0.8280231356620789


Forward chain:

$$
w_{rbf}\rightarrow KAN\rightarrow\dot Y\rightarrow ODE\rightarrow\hat Y(t)\rightarrow L.
$$

Backward chain:

$$
L\rightarrow\hat Y(t)\rightarrow ODE\rightarrow KAN\rightarrow\nabla_{w_{rbf}}L.
$$

## Step 16 · Verify one Adam update

In [281]:
optimizer = torch.optim.Adam(core.parameters(), lr=2e-3)

before = core.add.edges.w_rbf.detach().clone()

optimizer.zero_grad(set_to_none=True)
pred = integrate(dynamics, Y0, t_small, solver)
loss = trajectory_mse(loss_norm.normalize(pred), target_norm)
loss.backward()
optimizer.step()

after = core.add.edges.w_rbf.detach().clone()
delta = (after-before).norm()

print('update norm    :', delta.item())
print('before==after? :', torch.allclose(before,after))
assert delta > 0

update norm    : 0.018330171704292297
before==after? : False


## Step 17 · Inspect the real generic training loop

In [282]:
import chemkan.training as training_module

print('train_kinetic_stage:')
print(inspect.getsource(train_kinetic_stage))

print('\n_shared _optimize:')
print(inspect.getsource(training_module._optimize))

train_kinetic_stage:
def train_kinetic_stage(kinetic_dynamics: torch.nn.Module, y0: torch.Tensor,
                        t: torch.Tensor, loss_fn: LossFn, *, epochs: int,
                        lr: float, solver: SolverConfig,
                        log_every: int = 100) -> float:
    r"""Integrate species Y (temperature supplied externally); update the kinetic core.

        y0 : (B, m)   t : (T,)   loss_fn : (T, B, m) -> scalar

    ``lr`` (a paper experiment choice) and ``solver`` (explicit PyTorch settings) are
    required -- the reusable library never invents them. Optimizes only
    ``kinetic_dynamics.kinetic.parameters()``.
    """
    return _optimize(kinetic_dynamics, y0, t,
                     kinetic_dynamics.kinetic.parameters(),
                     loss_fn, epochs=epochs, lr=lr, solver=solver, log_every=log_every)


_shared _optimize:
def _optimize(func: torch.nn.Module, y0: torch.Tensor, t: torch.Tensor,
              params, loss_fn: LossFn, *, epochs: int, lr: flo

Fill this mapping yourself:

| Concept | Where is it in `_optimize`? |
|---|---|
| create Adam optimizer | |
| clear gradients | |
| integrate ODE | |
| compute loss | |
| backward pass | |
| update parameters | |
| logging | |

### MY NOTES

## Step 18 · Optional 2-epoch smoke call using the real training function

This is not a paper run. It only verifies that you understand how the reusable training function receives the pieces.

In [283]:
def tiny_loss_fn(pred_phys):
    return trajectory_mse(loss_norm.normalize(pred_phys), target_norm)

final_tiny_loss = train_kinetic_stage(
    dynamics,
    Y0,
    t_small,
    tiny_loss_fn,
    epochs=2,
    lr=2e-3,
    solver=solver,
    log_every=1,
)

print('tiny final loss:', final_tiny_loss)

tiny final loss: 0.008302874863147736


# Part VII · Review `train_biodiesel.py`

Now open the real script and locate this chain:

```text
load biodiesel
↓
create KineticCore
↓
create full-state normalizer
↓
ConstantTemperature
↓
KineticDynamics
↓
create species loss target
↓
define loss_fn(pred)
↓
train_kinetic_stage
↓
save checkpoint + metadata
```

Do not spend equal time on every utility line. Focus on data flow, shapes, physical meaning, and trainable parameters.

# Part VIII · PINN loss before hydrogen

The element-conservation loss uses physical species mass fractions.

The production contraction is

$$
z_{t,b,i}=\sum_k Y_{t,b,k}\,C_{i,k},
$$

implemented as

```python
torch.einsum('tbk,ik->tbi', Y_phys, coeff)
```

Here:

- `t` = time,
- `b` = trajectory,
- `k` = species and is summed away,
- `i` = chemical element and is retained.

In [284]:
print(inspect.getsource(element_conservation_loss))

def element_conservation_loss(Y_phys: torch.Tensor, element_counts: torch.Tensor,
                              atomic_weights: torch.Tensor, molar_weights: torch.Tensor
                              ) -> torch.Tensor:
    r"""Eq. 18 PINN term on PHYSICAL species mass fractions.

    Elemental mass fraction of element i:  z_i = sum_k N_i^k * W_i * Y_k / W_k.
    Penalise its drift from the initial state, summed over elements and timesteps,
    averaged over trajectories.

        Y_phys         : (T, B, m)  physical (denormalized) mass fractions
        element_counts : (Ne, m)    N_i^k  (atoms of element i in species k)
        atomic_weights : (Ne,)      W_i
        molar_weights  : (m,)       W_k
    """
    coeff = element_counts * atomic_weights[:, None] / molar_weights[None, :]   # (Ne, m)
    z = torch.einsum("tbk,ik->tbi", Y_phys, coeff)             # (T, B, Ne) elemental mass
    drift = (z - z[0:1]).abs()                                 # vs initial state -> (T,B,Ne)
    retu

### CHECKPOINT 8

Compare the Einstein logic:

```text
KAN : bik,oik -> boi   # k = RBF basis, summed
PINN: tbk,ik  -> tbi   # k = species, summed
```

The **letter** has no permanent meaning; the indices are defined by the equation/context.

### MY NOTES

# Part IX · Hydrogen roadmap

Do this after the biodiesel path above feels obvious.

## Stage 1

Expected flow:

```text
physical Y(t)              (B,9)
observed T(t)               (B,1)
        ↓ concatenate
physical [Y,T]              (B,10)
        ↓ minmax copy
model input                 (B,10)
        ↓ KineticCore
physical dY/dt              (B,9)
        ↓ solver
predicted Y trajectory      (T,B,9)
```

Inspect the real interpolation function:

In [285]:
print(inspect.getsource(ObservedTemperature.forward))

    def forward(self, t: torch.Tensor) -> torch.Tensor:      # scalar t -> (B, 1)
        times = self.saved_times
        t = torch.as_tensor(t, dtype=times.dtype, device=times.device)
        t = t.clamp(times[0], times[-1])
        # right-bracket index in [1, T-1], kept as a tensor to avoid a per-step
        # device->host sync (no int()/.item()); index_select gathers on-device.
        hi = torch.searchsorted(times, t.reshape(1), right=True).clamp(1, times.numel() - 1)
        lo = hi - 1
        t0, t1 = times.index_select(0, lo), times.index_select(0, hi)   # each (1,)
        w = ((t - t0) / (t1 - t0)).view(1, 1, 1)                        # weight in [0, 1]
        T0 = self.temperatures.index_select(0, lo)                      # (1, B, 1)
        T1 = self.temperatures.index_select(0, hi)
        return (T0 + w * (T1 - T0)).squeeze(0)                          # (B, 1)



## Stage 2

Now the ODE state itself is the full physical state:

$$
u=[Y_1,\ldots,Y_9,T].
$$

The thermodynamic branch is

$$
\dot T=Linear(\dot Y)+KAN_{cor}(u).
$$

In [286]:
print('ThermodynamicSuperstructure.forward:')
print(inspect.getsource(ThermodynamicSuperstructure.forward))

print('\nChemKAN.forward:')
print(inspect.getsource(ChemKAN.forward))

print('\nChemKANDynamics.forward:')
print(inspect.getsource(ChemKANDynamics.forward))

ThermodynamicSuperstructure.forward:
    def forward(self, u: torch.Tensor, dYdt: torch.Tensor) -> torch.Tensor:
        # u: (B, m+1), dYdt: (B, m) -> dT/dt: (B, 1)
        return self.linear(dYdt) + self.correction(u)


ChemKAN.forward:
    def forward(self, u: torch.Tensor) -> torch.Tensor:  # (B, m+1) -> (B, m+1)
        dYdt = self.kinetic(u)
        dTdt = self.thermo(u, dYdt)
        return torch.cat([dYdt, dTdt], dim=-1)


ChemKANDynamics.forward:
    def forward(self, t: torch.Tensor, u_physical: torch.Tensor) -> torch.Tensor:
        # (B, m+1) physical -> (B, m+1) physical derivative
        u_model = (u_physical if self.input_normalizer is None
                   else self.input_normalizer.normalize(u_physical))
        return self.model(u_model)



### Hydrogen checks to do next

1. `ObservedTemperature(t)` returns `(B,1)`.
2. Stage 1 integrates only 9 species.
3. Stage 1 KAN input is `(B,10)`.
4. Stage 2 ODE state is `(B,10)`.
5. `ThermodynamicSuperstructure` returns `(B,1)`.
6. full `ChemKAN` returns `(B,10)`.
7. Stage 2 optimizes all model parameters.
8. MSE uses normalized states.
9. elemental conservation uses physical species values.
10. species ordering matches the element-count matrix.

Do not tackle FSA until this forward/training chain is clear.

# Final self-check

You are ready to stop dissecting this implementation when you can answer these without guessing:

1. What does every axis of `(20,30,6)` mean?
2. Why is temperature appended before the kinetic core?
3. What is the difference between physical ODE state, min-max model input, and internal `tanh` input?
4. What does `(4,7,3)` mean for the first `w_rbf`?
5. What does `KineticCore` output physically?
6. What does `KineticDynamics` add around `KineticCore`?
7. Why can the ODE solver call the model more times than there are stored observations?
8. Why is loss computed on the integrated trajectory?
9. How does `loss.backward()` reach `w_rbf`?
10. What does `_optimize` add beyond the pieces you manually ran?
11. What changes in hydrogen Stage 1?
12. What changes in Stage 2?
13. Why must elemental conservation use physical species values?

Afterward, use an AI agent mainly to clean explanations, add references, and improve presentation.